# Module 1 — Chuẩn hóa một bảng tài chính
Chọn kernel **Colab** trong VS Code hoặc chạy trên web Colab. Notebook này dùng bundle đã import, bắt đầu với bảng kết quả kinh doanh BTS/2025 trang 12. Chạy lần lượt từ trên xuống. Không cần GPU, không đọc ảnh cho đến khi bật xem crop. Code không chạy phép cân đối kế toán hoặc C1.


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
root = Path('/content/drive/MyDrive')
if not root.exists():
    root = Path('/content/drive/My Drive')
assert root.is_dir(), 'Drive is not mounted'


## Lấy code mới
Giữ code notebook cá nhân ở máy bạn; repository dưới `/content` lấy từ GitHub và chỉ chứa code. Ô này reload module để tránh giữ code cũ trong kernel.


In [ ]:
import importlib
import os
import subprocess
import sys

repo = Path('/content/Mutil_Agent_Financial_Report')
url = 'https://github.com/VuongQuocAn/Mutil_Agent_Financial_Report.git'
if (repo / '.git').is_dir():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', url, str(repo)], check=True)
os.chdir(repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
importlib.invalidate_caches()
import financial_multiagent.ingestion.ocr_quality as quality
import financial_multiagent.ingestion.financial_normalization as normalization
import financial_multiagent.ingestion.precomputed_ocr as adapter
importlib.reload(quality)
importlib.reload(normalization)
importlib.reload(adapter)
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)


## Chọn bundle và bảng
Giữ BTS/2025 trang 12 cho lượt đầu. Nếu chọn báo cáo khác, cần xác nhận bảng có tên kết quả kinh doanh và header chỉ tiêu/mã số/năm nay/năm trước. Các layout khác sẽ báo lỗi rõ ràng.


In [ ]:
TICKER = 'BTS'
YEAR = '2025'
ACCESS_SCOPE = 'demo-scope'
PAGE = 12
TABLE_ID = 'table_001'

bundle_dir = root / 'Financial_MultiAgent_Results' / TICKER / YEAR / 'ocr-bundle-v1'
assert (bundle_dir / 'manifest.json').is_file(), 'Imported bundle missing: check bundle_dir'
print('Bundle:', bundle_dir)


## Metadata được xác nhận
Company ID dưới đây là lựa chọn của người chạy cho bundle, không phải mapping sang mã pháp nhân. Mặc định scope và ngày đầu kỳ để trống; chương trình nhận ngày cuối kỳ từ tiêu đề nếu đủ rõ nhưng không tự gán ngày đầu kỳ là 1/1. Khi đã đối chiếu tài liệu, điền metadata cùng ghi chú nguồn; metadata mâu thuẫn với tiêu đề sẽ được giữ ở trạng thái review.

Ví dụ cấu trúc (chỉ điền sau khi xác nhận): `PERIOD_OVERRIDES = {3: {"start": "YYYY-MM-DD", "end": "YYYY-MM-DD", "evidence": "trang/đoạn đã đối chiếu"}, 4: {...}}`. Trang 12 BTS hiện có cột 3 năm nay, cột 4 năm trước. EPS có đơn vị riêng; không áp dụng hệ số triệu/nghìn của toàn bảng cho EPS. Dấu `_`, dấu gạch hoặc ô trống giữ missing, không đổi thành 0.


In [ ]:
COMPANY_ID = TICKER
CONSOLIDATION_SCOPE = None  # 'standalone' or 'consolidated' only after confirmation
SCOPE_EVIDENCE = None
PERIOD_OVERRIDES = {}
EPS_UNIT = None  # 'VND/share' only after confirming the EPS unit
EPS_UNIT_EVIDENCE = None
ACCOUNTING_BASIS = None  # 'VAS' or 'IFRS' only after confirmation
BASIS_EVIDENCE = None

context = normalization.NormalizationContext(
    company_id=COMPANY_ID,
    company_evidence='Runner-selected company binding for this bundle',
    consolidation_scope=CONSOLIDATION_SCOPE, scope_evidence=SCOPE_EVIDENCE,
    period_overrides=PERIOD_OVERRIDES,
    eps_unit=EPS_UNIT, eps_unit_evidence=EPS_UNIT_EVIDENCE,
    accounting_basis=ACCOUNTING_BASIS, basis_evidence=BASIS_EVIDENCE,
)


## Tạo FinancialFact candidates
Đầu ra vào `normalization/<run-id>/` gồm facts JSONL, issue JSONL và summary JSON. Bước này chạy lại các kiểm chất lượng JSON hiện có và giữ các cảnh báo liên quan theo nguồn. Nhận xét chung rằng nội dung OCR ổn không tự xác nhận mọi ô. Fact có đủ ngữ cảnh mang nhãn `normalized_candidate`, còn `needs_review` ghi rõ thiếu metadata hoặc OCR issue. Cả hai chưa phải claim đã xác minh hay document ready.


In [ ]:
report = normalization.normalize_financial_table(
    bundle_dir, page=PAGE, table_id=TABLE_ID,
    access_scope=ACCESS_SCOPE, context=context,
)
run_dir = Path(report['output_dir'])
print('Execution:', report['execution_status'])
print('Quality:', report['quality_status'])
print('Facts:', report['fact_count'])
print('Parsed numeric values:', report['parsed_value_count'])
print('Canonical values:', report['canonical_value_count'])
print('Fact states:', report['fact_state_counts'])
print('Issue counts:', report['issue_counts'])
print('Detected context:', report['detected_context'])
print('Output:', run_dir)


## Xem dữ kiện chuẩn hóa
Giá trị chuẩn `value` ở đơn vị cơ sở `VND` hoặc `VND/share`; `parsed_value` giữ số trước khi nhân hệ số. Các giá trị đều là chuỗi thập phân. Chỉ tiêu dùng ontology nội bộ có phiên bản, chưa phải mapping VAS/IFRS đã được thẩm định.


In [ ]:
import json

facts = [json.loads(line) for line in (run_dir / 'financial_facts.jsonl').read_text(encoding='utf-8').splitlines()]
for fact in facts[:12]:
    print(fact['statement_code'], fact['metric_code'], fact['column_role'],
          'raw=', fact['raw_value'], 'value=', fact['value'], fact['unit'],
          'period=', fact['period_start'], fact['period_end'],
          'state=', fact['quality_state'])
    print('  reasons:', fact['normalization_issues'])


## Truy ngược một fact về ảnh — tùy chọn
Mặc định xem doanh thu thuần năm nay. Chỉ bật `SHOW_IMAGE` khi cần; crop hiển thị sẽ truyền qua kết nối của bạn. Có thể đối chiếu cả label/header/unit qua `context_refs`. Nếu ảnh nguồn đã đổi hoặc bị di chuyển sau import, cần xử lý lại nguồn/version trước khi dùng kết quả như bằng chứng hiện hành.


In [ ]:
SHOW_IMAGE = False
selected = next((f for f in facts if f['metric_code'] == 'revenue_net' and f['column_role'] == 'current'), facts[0])
print(json.dumps(selected, ensure_ascii=False, indent=2))
record, page_image = adapter.resolve_source_ref(
    bundle_dir, root, selected['source_ref'], access_scope=ACCESS_SCOPE)
print('Source:', page_image)
print('Raw source text:', record.get('text'))
if SHOW_IMAGE:
    from PIL import Image
    from IPython.display import display
    with Image.open(page_image) as image:
        display(image.crop(tuple(round(x) for x in record['bbox'])))
